# Ferry OD map
Same Folium curved OD-route plot as the original notebook, adapted to the ferry OD matrix.

In [ ]:

import pandas as pd
import folium
import numpy as np
import branca.colormap as cm

# -------------------------
# 1 Load ferry OD matrix
# -------------------------

raw = pd.read_excel("Faerge_OD_matrix_2025.xlsx", sheet_name="OD_Matrix_2025_Total", header=None)

destinations = raw.iloc[3, 1:].tolist()
destinations = [d for d in destinations if pd.notna(d)]

matrix = raw.iloc[4:, :len(destinations)+1].copy()
matrix.columns = ["origin"] + destinations
matrix = matrix[matrix["origin"].notna()]
matrix = matrix[matrix["origin"] != "Total in"]
matrix = matrix.set_index("origin")

if "Total out" in matrix.columns:
    matrix = matrix.drop(columns=["Total out"])

od_matrix = matrix.apply(pd.to_numeric, errors="coerce").fillna(0)

print("OD matrix used for map:")
print(od_matrix)

# -------------------------
# 2 Ferry port coordinates
# -------------------------
# Approximated port / ferry terminal coordinates used to preserve
# the exact same map style as the original notebook.

port_coords = {
    "Aarhus": [
        56.1496,
        10.2134
    ],
    "Aarø": [
        55.257,
        9.73
    ],
    "Aarøsund": [
        55.2622,
        9.7122
    ],
    "Agersø": [
        55.222,
        11.183
    ],
    "Agger": [
        56.781,
        8.234
    ],
    "Anholt": [
        56.716,
        11.511
    ],
    "Askø": [
        54.888,
        11.179
    ],
    "Assens": [
        55.27,
        9.9
    ],
    "Baagø": [
        55.223,
        9.786
    ],
    "Ballebro": [
        54.94,
        9.845
    ],
    "Bandholm": [
        54.836,
        11.487
    ],
    "Barsø": [
        55.116,
        9.576
    ],
    "Barsø Landing": [
        55.116,
        9.59
    ],
    "Bogø": [
        54.928,
        12.05
    ],
    "Branden": [
        56.845,
        8.975
    ],
    "Bøjden": [
        55.078,
        10.079
    ],
    "Christiansø": [
        55.32,
        15.187
    ],
    "Ebeltoft": [
        56.194,
        10.683
    ],
    "Egense": [
        57.048,
        10.268
    ],
    "Endelave": [
        55.756,
        10.304
    ],
    "Esbjerg": [
        55.467,
        8.451
    ],
    "Fanø": [
        55.44,
        8.4
    ],
    "Fejø": [
        54.953,
        11.396
    ],
    "Femø": [
        54.985,
        11.54
    ],
    "Frederikshavn": [
        57.44,
        10.536
    ],
    "Fur": [
        56.823,
        8.974
    ],
    "Fynshav": [
        54.9861,
        9.9814
    ],
    "Fåborg": [
        55.095,
        10.242
    ],
    "Grenaa": [
        56.415,
        10.927
    ],
    "Gudhjem": [
        55.209,
        14.971
    ],
    "Hals": [
        56.996,
        10.308
    ],
    "Hardeshøj": [
        55.002,
        9.998
    ],
    "Havnsø": [
        55.751,
        11.302
    ],
    "Holbæk": [
        55.718,
        11.712
    ],
    "Horsens": [
        55.86,
        9.85
    ],
    "Hov": [
        55.926,
        10.262
    ],
    "Hundested": [
        55.963,
        11.851
    ],
    "Hvalpsund": [
        56.798,
        9.201
    ],
    "Kalundborg": [
        55.679,
        11.089
    ],
    "Kleppen": [
        56.824,
        8.246
    ],
    "Kragenæs": [
        54.928,
        11.368
    ],
    "Køge": [
        55.458,
        12.182
    ],
    "Læsø": [
        57.255,
        10.995
    ],
    "Marstal": [
        54.855,
        10.517
    ],
    "Mommark": [
        54.932,
        10.022
    ],
    "Omø": [
        55.157,
        11.151
    ],
    "Orø": [
        55.752,
        11.822
    ],
    "Rudkøbing": [
        54.939,
        10.713
    ],
    "Rønne": [
        55.101,
        14.706
    ],
    "Rørvig": [
        55.943,
        11.776
    ],
    "Samsø": [
        55.856,
        10.617
    ],
    "Sejerø": [
        55.891,
        11.143
    ],
    "Sjællands Odde": [
        55.979,
        11.367
    ],
    "Spodsbjerg": [
        54.935,
        10.835
    ],
    "Stigsnæs": [
        55.223,
        11.273
    ],
    "Strynø": [
        54.901,
        10.621
    ],
    "Stubbekøbing": [
        54.888,
        12.041
    ],
    "Sundsøre": [
        56.625,
        9.17
    ],
    "Svendborg": [
        55.059,
        10.607
    ],
    "Sælvig": [
        55.833,
        10.668
    ],
    "Søby": [
        54.938,
        10.255
    ],
    "Thyborøn": [
        56.698,
        8.219
    ],
    "Tunø": [
        55.952,
        10.458
    ],
    "Tårs": [
        54.894,
        11.019
    ],
    "Udbyhøj Nord": [
        56.609,
        10.329
    ],
    "Udbyhøj Syd": [
        56.605,
        10.329
    ],
    "Venø": [
        56.548,
        8.644
    ],
    "Ærøskøbing": [
        54.888,
        10.411
    ]
}

# sanity check
missing_ports = sorted(set(od_matrix.index).union(set(od_matrix.columns)) - set(port_coords.keys()))
if missing_ports:
    raise ValueError(f"Missing coordinates for ports: {missing_ports}")

# -------------------------
# 3 Rounded curve function
# -------------------------

def rounded_route(start, end, bend=0.25, n_points=40):
    lat1, lon1 = start
    lat2, lon2 = end

    dx = lon2 - lon1
    dy = lat2 - lat1

    mid_lat = (lat1 + lat2) / 2
    mid_lon = (lon1 + lon2) / 2

    offset_lat = -dy * bend
    offset_lon = dx * bend

    ctrl_lat = mid_lat + offset_lat
    ctrl_lon = mid_lon + offset_lon

    points = []

    for t in np.linspace(0, 1, n_points):
        lat = (1 - t) ** 2 * lat1 + 2 * (1 - t) * t * ctrl_lat + t ** 2 * lat2
        lon = (1 - t) ** 2 * lon1 + 2 * (1 - t) * t * ctrl_lon + t ** 2 * lon2
        points.append((lat, lon))

    return points

# -------------------------
# 4 Log scaling
# -------------------------

flows = od_matrix.to_numpy().astype(float).flatten()
flows = flows[flows > 0]

min_log = np.log1p(flows.min())
max_log = np.log1p(flows.max())

def normalize(x):
    if max_log == min_log:
        return 0.5
    return (np.log1p(x) - min_log) / (max_log - min_log)

# -------------------------
# 5 Create map
# -------------------------

m = folium.Map(location=[56.2, 10.0], zoom_start=7)

# -------------------------
# 6 Draw ports
# -------------------------

for port, coord in port_coords.items():
    folium.CircleMarker(
        location=coord,
        radius=6,
        color="black",
        fill=True,
        fill_color="white",
        fill_opacity=1,
        popup=port
    ).add_to(m)

# -------------------------
# 7 Red color scale
# -------------------------

colormap = cm.linear.Reds_09.scale(0, 1)

# -------------------------
# 8 Draw OD routes
# -------------------------

for origin in od_matrix.index:
    for destination in od_matrix.columns:
        value = od_matrix.loc[origin, destination]

        if value <= 0:
            continue

        if origin == destination:
            continue

        start = port_coords.get(origin)
        end = port_coords.get(destination)

        if start is None or end is None:
            continue

        norm_val = normalize(value)
        curve = rounded_route(start, end)

        folium.PolyLine(
            curve,
            color=colormap(norm_val),
            weight=2 + 3 * norm_val,
            opacity=0.9,
            dash_array="5,8",
            popup=f"{origin} → {destination}: {value:,.1f} (1,000 passengers)"
        ).add_to(m)

# -------------------------
# 9 Add legend
# -------------------------

colormap.caption = "Passenger flow (log scale, 1,000 passengers)"
colormap.add_to(m)

# -------------------------
# 10 Save map
# -------------------------

m.save("denmark_ferry_od_network_logscale.html")

print("Map saved as denmark_ferry_od_network_logscale.html")
